# 05 — Zielgruppenvalidierung mit WEDA-FALL

Entspricht Kapitel 5, Abschnitt *Zielgruppe*.

Alle bisher verwendeten Datensaetze bestehen ueberwiegend aus jungen
Probanden. WEDA-FALL enthaelt zusaetzlich 11 Personen zwischen 77 und
95 Jahren — die tatsaechliche Zielgruppe des Vorhabens.

**Drei Besonderheiten des Datensatzes:**

1. Die aelteren Teilnehmer haben aus Sicherheitsgruenden **keine
   Stuerze** ausgefuehrt (nur D01, D03, D04, D09, D10, D11). Fuer
   diese Gruppe ist ausschliesslich die Fehlalarmrate bewertbar.
2. Die Zeitstempel des Fitbit Sense sind ungleichmaessig (Buendel im
   Abstand von 1–2 ms, Luecken bis 93 ms; effektiv 40–50 Hz). Das
   Lademodul interpoliert auf ein regelmaessiges Raster.
3. `dataset/` enthaelt **fuenf** Abtastraten (5, 10, 25, 40, 50 Hz).
   Kopiert wird nur `50Hz` — ein Fuenftel der Dateien. Das Kopieren
   des gesamten Ordners ueber die Drive-Einbindung schlaegt bei
   zehntausenden Kleindateien haeufig fehl.

**Zur Auswertung:** Ein Vergleich beider Gruppen ueber alle
Aktivitaeten waere unfair, da die Aelteren nur eine Teilmenge
ausgefuehrt haben. Der Vergleich beschraenkt sich daher auf die
gemeinsamen Aktivitaeten.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

!wget -q https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py -O /content/fall_lib.py
sys.path.insert(0, "/content")
import fall_lib as fl

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"

print("Konfiguration:", fl.default_config())

Mounted at /content/drive
Konfiguration: {'fs_target': 50, 'window_sec': 2.0, 'overlap': 0.5, 'window_len': 100, 'step': 50, 'kfall_fall_tasks': 'T20-T34'}


## Datensatz kopieren

Nur der 50-Hz-Zweig wird kopiert. Nach dem Kopieren wird geprueft,
ob die erwartete Zahl an Dateien angekommen ist — ein unvollstaendiger
Kopiervorgang faellt sonst erst viel spaeter und mit unklarer
Fehlermeldung auf.

In [2]:
QUELLE = f"{BASE}/WEDA-FALL-main/dataset/50Hz"
ZIEL   = "/content/WEDA50"

if not Path(QUELLE).exists():
    raise FileNotFoundError(
        f"{QUELLE} nicht gefunden.\n"
        f"  Inhalt von {BASE}/WEDA-FALL-main/dataset:\n    "
        + "\n    ".join(p.name for p in
                        sorted(Path(f"{BASE}/WEDA-FALL-main/dataset").iterdir())))

# Unvollstaendige Reste einer frueheren Ausfuehrung entfernen
if Path(ZIEL).exists():
    n_alt = len(list(Path(ZIEL).glob("*/*.csv")))
    print(f"[i] {ZIEL} existiert bereits mit {n_alt} CSV-Dateien")
    if n_alt < 3000:
        print("[i] unvollstaendig -> wird neu kopiert")
        shutil.rmtree(ZIEL)

if not Path(ZIEL).exists():
    print("Kopiere 50-Hz-Daten ...")
    t = time.time()
    shutil.copytree(QUELLE, ZIEL)
    print(f"  fertig in {time.time()-t:.0f}s")

# Kontrolle
ordner = sorted(p.name for p in Path(ZIEL).iterdir() if p.is_dir())
n_csv  = len(list(Path(ZIEL).glob("*/*.csv")))
print(f"\nBewegungsordner ({len(ordner)}): {ordner}")
print(f"CSV-Dateien: {n_csv}")

if len(ordner) < 19 or n_csv < 3000:
    raise RuntimeError(
        f"Kopie unvollstaendig: {len(ordner)} Ordner / {n_csv} Dateien.\n"
        f"  Erwartet: 19 Ordner (D01-D11, F01-F08), rund 3900 Dateien.\n"
        f"  Zelle erneut ausfuehren — Drive bricht bei vielen "
        f"Kleindateien gelegentlich ab.")
print("Kopie vollstaendig.")

Kopiere 50-Hz-Daten ...
  fertig in 106s

Bewegungsordner (19): ['D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08']
CSV-Dateien: 3719
Kopie vollstaendig.


## Modell laden

In [3]:
from tensorflow import keras

if not Path(MODEL).exists():
    raise FileNotFoundError(
        f"{MODEL} fehlt — bitte zuerst 01_basismodelle ausfuehren.")

model = keras.models.load_model(MODEL, compile=False)
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
print(f"Modell geladen: {model.count_params():,} Parameter")

run = fl.new_run("05_zielgruppe")
ergebnisse = {}

Modell geladen: 61,601 Parameter
[Lauf] /content/drive/MyDrive/ML_Fall_Detection/results/05_zielgruppe_2026-08-26_1148


## Trials laden

`find_weda_trials` sucht zunaechst einen Unterordner `50Hz`. Da hier
bereits direkt auf diesen Ordner gezeigt wird, greift die
Rueckfallebene und der Ordner selbst dient als Wurzel.

In [4]:
trials_alle = fl.find_weda_trials(ZIEL)
if not trials_alle:
    raise RuntimeError(
        f"Keine Trials erkannt. Erwartetes Schema: "
        f"<Bewegungscode>/U01_R01_accel.csv")

from collections import Counter
print(f"Trials gesamt: {len(trials_alle)}")
print(f"  nach Gruppe:  {dict(Counter(e['group'] for e in trials_alle))}")
print(f"  nach Klasse:  "
      f"{dict(Counter('Sturz' if e['label'] else 'ADL' for e in trials_alle))}")
print("\nErwartet laut Dokumentation: 350 Sturz / 619 ADL, "
      "davon 157 Trials von aelteren Probanden.")

Trials gesamt: 969
  nach Gruppe:  {'young': 812, 'elderly': 157}
  nach Klasse:  {'ADL': 619, 'Sturz': 350}

Erwartet laut Dokumentation: 350 Sturz / 619 ADL, davon 157 Trials von aelteren Probanden.


In [5]:
alt  = fl.load_weda(ZIEL, group="elderly")
jung = fl.load_weda(ZIEL, group="young")

if not alt or not jung:
    raise RuntimeError(
        f"Eine Gruppe ist leer (alt={len(alt)}, jung={len(jung)}). "
        f"Pruefen Sie die Zuordnung in fl.WEDA_ELDERLY / fl.WEDA_YOUNG.")

[WEDA elderly] 157 Trials (0 Sturz, 157 ADL), 11 Probanden
[WEDA young] 812 Trials (350 Sturz, 462 ADL), 14 Probanden


## Fairer Gruppenvergleich

Beschraenkt auf die Aktivitaeten, die beide Gruppen ausgefuehrt
haben.

In [6]:
akt_alt   = {t["activity"] for t in alt}
akt_jung  = {t["activity"] for t in jung}
gemeinsam = sorted(akt_alt & akt_jung)

print(f"Aeltere ({len(akt_alt)}):   {sorted(akt_alt)}")
print(f"Junge   ({len(akt_jung)}):  {sorted(akt_jung)}")
print(f"Gemeinsam ({len(gemeinsam)}): {gemeinsam}")

if not gemeinsam:
    raise RuntimeError("Keine gemeinsamen Aktivitaeten — "
                       "Vergleich nicht moeglich.")

SEK_PRO_FENSTER = fl.WIN_SEC * (1 - fl.OVERLAP)

def fp_analyse(trials, aktivitaeten, name):
    """Fehlalarmrate auf ADL-Aufzeichnungen der gegebenen
    Aktivitaeten. Da keine Stuerze enthalten sind, ist jede positive
    Klassifikation ein Fehlalarm."""
    sub = [t for t in trials
           if t["activity"] in aktivitaeten and t["label"] == 0]
    if not sub:
        raise ValueError(f"{name}: keine passenden ADL-Trials")

    X, y, g, a = fl.build_windows(sub, mode="sliding")
    if len(X) == 0:
        raise ValueError(f"{name}: keine Fenster erzeugt")

    p = model.predict(fl.znorm(X), batch_size=256, verbose=0).flatten()
    h = len(X) * SEK_PRO_FENSTER / 3600
    print(f"  {name:<10} {len(X):>6} Fenster = {h:>5.2f} Std, "
          f"{len(np.unique(g)):>2} Probanden")
    return p, a, h

print()
p_alt,  a_alt,  h_alt  = fp_analyse(alt,  gemeinsam, "Aeltere")
p_jung, a_jung, h_jung = fp_analyse(jung, gemeinsam, "Junge")

Aeltere (6):   ['D01', 'D03', 'D04', 'D09', 'D10', 'D11']
Junge   (19):  ['D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08']
Gemeinsam (6): ['D01', 'D03', 'D04', 'D09', 'D10', 'D11']

  Aeltere      1469 Fenster =  0.41 Std, 11 Probanden
  Junge        3521 Fenster =  0.98 Std, 14 Probanden


### Fehlalarmrate nach Entscheidungsschwelle

In [7]:
print(f"{'Schwelle':>9} {'FP alt':>8} {'/Std':>8} "
      f"{'FP jung':>9} {'/Std':>8} {'Verhaeltnis':>12}")
print("-"*60)

vergleich = {}
for thr in [0.5, 0.7, 0.9, 0.95, 0.99]:
    fa, fj = int((p_alt > thr).sum()), int((p_jung > thr).sum())
    ra, rj = fa / h_alt, fj / h_jung
    q = ra / rj if rj > 0 else float("nan")
    print(f"{thr:>9.2f} {fa:>8d} {ra:>8.1f} {fj:>9d} {rj:>8.1f} "
          f"{q:>11.2f}x")
    vergleich[str(thr)] = {
        "elderly_fp": fa, "elderly_per_hour": round(ra, 2),
        "young_fp": fj,   "young_per_hour": round(rj, 2),
        "ratio": round(float(q), 3)}

ergebnisse["fairer_vergleich"] = {
    "aktivitaeten": gemeinsam,
    "elderly_hours": round(h_alt, 3),
    "young_hours": round(h_jung, 3),
    "schwellen": vergleich}

print("\nEin deutlich erhoehtes Verhaeltnis wuerde bedeuten, dass "
      "Alltagsbewegungen\naelterer Menschen fuer das auf jungen "
      "Kohorten trainierte Modell schwerer\nzu unterscheiden sind.")

 Schwelle   FP alt     /Std   FP jung     /Std  Verhaeltnis
------------------------------------------------------------
     0.50      121    296.5       112    114.5        2.59x
     0.70      103    252.4        92     94.1        2.68x
     0.90       82    201.0        52     53.2        3.78x
     0.95       76    186.2        43     44.0        4.24x
     0.99       58    142.1        25     25.6        5.56x

Ein deutlich erhoehtes Verhaeltnis wuerde bedeuten, dass Alltagsbewegungen
aelterer Menschen fuer das auf jungen Kohorten trainierte Modell schwerer
zu unterscheiden sind.


### Aufschluesselung nach Bewegung

Zeigt, ob ein etwaiger Gruppenunterschied auf einzelne Bewegungen
zurueckgeht oder durchgaengig auftritt.

In [8]:
print(f"{'Code':<6} {'Bewegung':<24} {'alt %':>8} {'jung %':>8} "
      f"{'Differenz':>11}")
print("-"*62)

nach_bewegung = {}
for mv in gemeinsam:
    pa, pj = p_alt[a_alt == mv], p_jung[a_jung == mv]
    if len(pa) == 0 or len(pj) == 0:
        continue
    ra, rj = 100 * (pa > 0.9).mean(), 100 * (pj > 0.9).mean()
    print(f"{mv:<6} {fl.WEDA_MOVEMENTS.get(mv, mv):<24} "
          f"{ra:>7.2f}% {rj:>7.2f}% {ra-rj:>+10.2f}pp")
    nach_bewegung[mv] = {
        "name": fl.WEDA_MOVEMENTS.get(mv, mv),
        "elderly_rate": round(ra / 100, 4),
        "young_rate": round(rj / 100, 4),
        "n_elderly": int(len(pa)), "n_young": int(len(pj))}

ergebnisse["nach_bewegung"] = nach_bewegung

Code   Bewegung                    alt %   jung %   Differenz
--------------------------------------------------------------
D01    Gehen                       0.70%    0.41%      +0.29pp
D03    Treppe auf/ab               0.81%    1.98%      -1.17pp
D04    Hinsetzen/Aufstehen         0.47%    0.00%      +0.47pp
D09    Auf Tisch schlagen          3.91%    5.19%      -1.29pp
D10    Klatschen                  25.51%    2.54%     +22.98pp
D11    Tuer oeffnen                4.35%    0.77%      +3.58pp


## Sensitivitaet bei den jungen Probanden

Nur diese Gruppe hat Stuerze ausgefuehrt. Der Wert dient als
Einordnung der Fehlalarmzahlen — eine niedrige Fehlalarmrate bei
gleichzeitig niedriger Sensitivitaet waere kein gutes Ergebnis.

In [9]:
X_j, y_j, g_j, a_j = fl.build_windows(jung, mode="standard")
if y_j.sum() > 0:
    ergebnisse["junge_gesamt"] = fl.evaluate(
        model, fl.znorm(X_j), y_j, "Junge (alle Aktivitaeten)")
    n_f = int(y_j.sum())
    print(f"\n  {n_f} Sturzfenster von "
          f"{len({t['subject'] for t in jung if t['label']==1})} Probanden")
else:
    print("Keine Sturzfenster vorhanden.")

  Junge (alle Aktivitaeten)   Se=0.254  Sp=0.964  Acc=0.923  FP=  205  FN= 261

  350 Sturzfenster von 14 Probanden


## Ergebnisse sichern

In [10]:
fl.save_results(
    run, ergebnisse, config=fl.default_config(),
    notes=("Aeltere Probanden ohne Stuerze -> nur Fehlalarmrate "
           "bewertbar. Gruppenvergleich auf gemeinsame Aktivitaeten "
           "beschraenkt. Nur 50-Hz-Zweig verwendet; Fitbit-"
           "Zeitstempel auf regelmaessiges Raster interpoliert."))

print()
print(json.dumps(ergebnisse.get("fairer_vergleich", {}), indent=2)[:1200])

[gesichert] /content/drive/MyDrive/ML_Fall_Detection/results/05_zielgruppe_2026-08-26_1148/results.json

{
  "aktivitaeten": [
    "D01",
    "D03",
    "D04",
    "D09",
    "D10",
    "D11"
  ],
  "elderly_hours": 0.408,
  "young_hours": 0.978,
  "schwellen": {
    "0.5": {
      "elderly_fp": 121,
      "elderly_per_hour": 296.53,
      "young_fp": 112,
      "young_per_hour": 114.51,
      "ratio": 2.589
    },
    "0.7": {
      "elderly_fp": 103,
      "elderly_per_hour": 252.42,
      "young_fp": 92,
      "young_per_hour": 94.06,
      "ratio": 2.683
    },
    "0.9": {
      "elderly_fp": 82,
      "elderly_per_hour": 200.95,
      "young_fp": 52,
      "young_per_hour": 53.17,
      "ratio": 3.78
    },
    "0.95": {
      "elderly_fp": 76,
      "elderly_per_hour": 186.25,
      "young_fp": 43,
      "young_per_hour": 43.96,
      "ratio": 4.236
    },
    "0.99": {
      "elderly_fp": 58,
      "elderly_per_hour": 142.14,
      "young_fp": 25,
      "young_per_hour": 25.56,